# Serving a Zarr array from a notebook

A notebook needs a server that outlives the cell that started it, which is the
one thing the `with serve_node(...)` form in the README cannot give you --
that shuts the server down as soon as the block ends.

The notebook pattern is instead:

1. start with `background=True` and keep the handle,
2. use the server across as many cells as you like,
3. `shutdown()` when you are done.

Two details make this comfortable in a kernel you re-run:

- **`background=True`** runs uvicorn in a daemon thread with its own event
  loop, so it never touches the kernel's loop and cannot block it.
- **`port=0`** asks the OS for a free port. Re-running a start cell without
  stopping the previous server is the classic notebook mistake; with a fixed
  port that fails with *address already in use*, while `port=0` simply picks
  another one. `server.url` reports the port actually bound.

In [ ]:
import numpy as np
import zarr
from zarr.storage import MemoryStore

from zarr_http_server import serve_node

store = MemoryStore()
array = zarr.create_array(
    store,
    data=np.arange(1000, dtype="uint8").reshape(10, 10, 10),
    chunks=(5, 5, 5),
    compressors=None,
    write_data=True,
)
array.info

## Start

`serve_node` returns a `BackgroundServer` as soon as the socket is listening,
so the next cell can use it immediately.

In [ ]:
server = serve_node(array, host="127.0.0.1", port=0, background=True)

print(f"serving at {server.url}")
assert server.url is not None

## Use it

The server is alive across cells now. Anything that speaks HTTP can read from
it -- here `httpx`, but a browser or another zarr client works the same way.

(With `fsspec[http]` installed you can also do
`zarr.open_array(server.url, mode="r")` to read the array back through zarr
itself.)

In [ ]:
import httpx

metadata = httpx.get(f"{server.url}/zarr.json", timeout=30)
print(metadata.status_code, metadata.headers["content-type"])
assert metadata.status_code == 200
assert metadata.json()["shape"] == [10, 10, 10]

In [ ]:
chunk = httpx.get(f"{server.url}/c/0/0/0", timeout=30)
print(f"chunk c/0/0/0: {len(chunk.content)} bytes")
assert chunk.status_code == 200

# Byte ranges work too, and say which bytes came back.
part = httpx.get(f"{server.url}/c/0/0/0", headers={"Range": "bytes=0-9"}, timeout=30)
print(part.status_code, part.headers["content-range"], part.content)
assert part.status_code == 206
assert part.content == chunk.content[:10]

## Writes are off unless you ask

The default is read-only, so a stray `PUT` from a notebook cell -- or from
anyone else who can reach the port -- is refused.

In [ ]:
refused = httpx.put(f"{server.url}/c/0/0/0", content=b"nope", timeout=30)
print("PUT ->", refused.status_code)
assert refused.status_code == 405

## Stop

`shutdown()` waits for in-flight requests, then forces the server closed. It
raises if the thread will not stop, so a silent failure cannot leave you
believing the port is free when it is not.

In [ ]:
server.shutdown()

# The port really is closed now.
try:
    httpx.get(f"{server.url}/zarr.json", timeout=5)
except httpx.HTTPError as exc:
    print(f"as expected, no longer serving: {type(exc).__name__}")
else:
    raise AssertionError("server still responding after shutdown")

## If you forget to stop one

The server thread is a daemon, so it dies with the kernel -- restarting the
kernel always clears it. Because `port=0` picks a fresh port each time, a
forgotten server does not block the next one either; it just holds a port
until the kernel exits.

If you want the shutdown tied to a block rather than a cell, the context
manager form still works inside a single cell:

```python
with serve_node(array, port=0, background=True) as server:
    ...  # everything must happen in this cell
```